# Tax AI V1.3 — Colab GPU + Gemma 4 E4B + api-ocr-2025

正式測試架構：**GitHub Pages → Vercel 免費 HTTPS Proxy → Colab GPU → api-ocr-2025 → Gemma 4 E4B**。

V1.3 改進：
- 修正 Colab `numpy / numba` 套件衝突；本流程只做影像，不安裝 `numba/librosa`。
- 固定 `numpy>=2.2,<2.3`、`requests==2.32.4`，降低 Colab相依漂移。
- 使用 Google/Hugging Face 官方 `AutoProcessor + AutoModelForMultimodalLM` 圖像路徑。
- Gemma sidecar 與模型在同一 Colab process，避免 subprocess 看不到已載入模型。
- `api-ocr-2025` 使用固定 commit `5ef5794c1b0c3fc640d6ac8c8d26562b6c035202`。
- 新增 `/v1/buyer-ban`：專門接收前端切好的 **8 格買受人統編**，只讀 8 碼，不用檢查碼猜字。
- 加入 CORS，Vercel Proxy 若超時，前端可自動直接連 Colab 備援。

安全規則：買受人 8 格=`buyer_tax_id`；右下專用章=`seller_tax_id`；統編檢查碼只驗證、不修改模型讀到的數字；不清楚就回 `null`。


In [ ]:
import sys, subprocess, torch
print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
subprocess.run(['nvidia-smi'], check=False)
if not torch.cuda.is_available():
    raise RuntimeError('請先在 Colab：執行階段 → 變更執行階段類型 → GPU')


## 1. 安裝相依套件

只保留影像辨識需要的套件。`numba/librosa` 是音訊路徑相依，本專案不使用，移除以避免與 api-ocr 所需 NumPy 2.2+ 衝突。


In [ ]:
!apt-get -qq update
!apt-get -qq install -y libzbar0
!pip -q uninstall -y numba librosa || true
!pip -q install -U "numpy>=2.2,<2.3" "requests==2.32.4" "transformers>=5.5.0" accelerate bitsandbytes huggingface_hub fastapi uvicorn python-multipart pillow


## 2. 下載並固定 api-ocr-2025


In [ ]:
import os, shutil, subprocess
REPO='/content/api-ocr-2025'
PIN='5ef5794c1b0c3fc640d6ac8c8d26562b6c035202'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','https://github.com/adi-gov-tw/api-ocr-2025.git',REPO],check=True)
subprocess.run(['git','checkout',PIN],cwd=REPO,check=True)
!pip -q install -r /content/api-ocr-2025/requirements.txt
!pip -q install "numpy>=2.2,<2.3" "requests==2.32.4"


In [ ]:
import numpy, requests, cv2, PIL, transformers
print('NumPy:', numpy.__version__)
print('requests:', requests.__version__)
print('OpenCV:', cv2.__version__)
print('Transformers:', transformers.__version__)
try:
    import torchvision
    print('torchvision:', torchvision.__version__)
except Exception as e:
    print('torchvision import warning:', repr(e))
try:
    import numba
    print('⚠ numba remains installed:', numba.__version__)
except Exception:
    print('✅ numba not installed — expected for image-only pipeline')


## 3. 載入 Gemma 4 E4B

免費 Colab 常見 T4 只有 16GB VRAM，因此優先用 bitsandbytes 4-bit。若 Hugging Face 要求授權，先在模型頁接受條款後重跑本格。


In [ ]:
import gc, torch
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig
MODEL_ID='google/gemma-4-E4B-it'
processor=AutoProcessor.from_pretrained(MODEL_ID)
major,_=torch.cuda.get_device_capability(0)
compute_dtype=torch.bfloat16 if major>=8 else torch.float16
qconfig=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)
model=AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=qconfig,
    device_map='auto',
    dtype=compute_dtype,
)
model.eval()
free,total=torch.cuda.mem_get_info()
print('✅ Gemma 4 E4B loaded in 4-bit')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM free/total GiB:', round(free/2**30,2), '/', round(total/2**30,2))


## 4. 啟動 OpenAI-compatible Gemma Vision sidecar (:8001)

與模型在同一 kernel/process 內啟動，並以 lock 序列化 GPU 推理。


In [ ]:
import base64, io, time, threading, requests, json, re
from fastapi import FastAPI, Request
from PIL import Image
import uvicorn

vlm_app=FastAPI(title='Gemma 4 E4B sidecar')
infer_lock=threading.Lock()

def decode_data_image(value):
    if isinstance(value,dict): value=value.get('url')
    if not isinstance(value,str): return None
    if value.startswith('data:') and ',' in value:
        return Image.open(io.BytesIO(base64.b64decode(value.split(',',1)[1]))).convert('RGB')
    return None

def collect_parts(messages):
    images=[]; texts=[]
    for msg in messages or []:
        content=msg.get('content','')
        if isinstance(content,str):
            texts.append(content)
        elif isinstance(content,list):
            for part in content:
                typ=part.get('type')
                if typ in ('text','input_text'):
                    texts.append(part.get('text',''))
                elif typ in ('image_url','input_image'):
                    val=part.get('image_url') or part.get('image')
                    img=decode_data_image(val)
                    if img is not None: images.append(img)
    return images, '\n'.join(texts)

def parsed_content(decoded):
    try:
        p=processor.parse_response(decoded)
        if isinstance(p,dict):
            c=p.get('content','')
            if isinstance(c,str): return c.strip()
            if isinstance(c,list):
                parts=[]
                for item in c:
                    if isinstance(item,str): parts.append(item)
                    elif isinstance(item,dict) and item.get('text'): parts.append(str(item['text']))
                return '\n'.join(parts).strip()
        return str(p).strip()
    except Exception:
        return re.sub(r'<\|[^>]+\|>','',decoded).strip()

@vlm_app.get('/health')
async def vlm_health():
    return {'status':'ok','model':MODEL_ID,'cuda':torch.cuda.is_available(),'mode':'same-process-thread'}

@vlm_app.get('/v1/models')
async def vlm_models():
    return {'object':'list','data':[{'id':MODEL_ID,'object':'model'}]}

@vlm_app.post('/v1/chat/completions')
async def vlm_chat(req:Request):
    body=await req.json()
    images,text=collect_parts(body.get('messages',[]))
    if images:
        messages=[{'role':'user','content':[{'type':'image','image':images[0]},{'type':'text','text':text or 'Describe this image.'}]}]
    else:
        messages=[{'role':'user','content':text or 'Reply OK'}]
    inputs=processor.apply_chat_template(
        messages, tokenize=True, return_dict=True, return_tensors='pt',
        add_generation_prompt=True, enable_thinking=False
    ).to(model.device)
    n=inputs['input_ids'].shape[-1]
    with infer_lock, torch.inference_mode():
        out=model.generate(
            **inputs,
            max_new_tokens=min(int(body.get('max_tokens') or 600),900),
            do_sample=False,
        )
    decoded=processor.decode(out[0][n:],skip_special_tokens=False)
    content=parsed_content(decoded)
    return {
      'id':'chatcmpl-gemma4e4b','object':'chat.completion','created':int(time.time()),'model':MODEL_ID,
      'choices':[{'index':0,'message':{'role':'assistant','content':content},'finish_reason':'stop'}]
    }

def run_vlm_server():
    uvicorn.Server(uvicorn.Config(vlm_app,host='127.0.0.1',port=8001,log_level='warning')).run()

vlm_thread=threading.Thread(target=run_vlm_server,daemon=True)
vlm_thread.start()
for _ in range(90):
    try:
        r=requests.get('http://127.0.0.1:8001/health',timeout=2)
        if r.ok:
            print('✅ VLM sidecar:',r.json());break
    except Exception: time.sleep(1)
else:
    raise RuntimeError('Gemma sidecar failed to start')


## 5. 加上 Tax AI 專用 API：CORS + 8 格買受人辨識

`/v1/invoice`、`/health/vlm` 仍由原版 api-ocr-2025 提供；只額外增加 `/v1/buyer-ban`。


In [ ]:
custom = r'''from app.main import app
import base64, json, re
from fastapi import File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from openai import OpenAI

app.add_middleware(
    CORSMiddleware,
    allow_origins=['https://alexandroslee.github.io'],
    allow_credentials=False,
    allow_methods=['GET','POST','OPTIONS'],
    allow_headers=['*'],
)

client=OpenAI(api_key='EMPTY',base_url='http://127.0.0.1:8001/v1')

def extract_json(text):
    text=(text or '').strip()
    text=re.sub(r'^```(?:json)?\s*|\s*```$','',text,flags=re.I|re.S).strip()
    m=re.search(r'\{.*\}',text,re.S)
    if not m: return {}
    try: return json.loads(m.group(0))
    except Exception: return {}

@app.post('/v1/buyer-ban')
async def buyer_ban(file:UploadFile=File(...)):
    data=await file.read()
    if not data: raise HTTPException(400,'空影像')
    if len(data)>3*1024*1024: raise HTTPException(413,'8格影像超過3MB')
    b64=base64.b64encode(data).decode()
    prompt=(
        'This image is a cropped Taiwan triplicate-invoice BUYER tax-ID strip containing exactly 8 cells from left to right. '
        'Read one digit from each cell. Ignore all borders/grid lines. '
        'Do NOT use Taiwan tax-ID checksum to guess, repair, substitute, or alter any digit. '
        'If any cell is genuinely unreadable, set buyer_tax_id to null. '
        'Return ONLY JSON: {"buyer_tax_id":"12345678" or null,"digits":["1","2","3","4","5","6","7","8"],"confidence":0.0}.'
    )
    try:
        resp=client.chat.completions.create(
            model='google/gemma-4-E4B-it',temperature=0,max_tokens=120,top_p=0.1,
            messages=[{'role':'user','content':[{'type':'image_url','image_url':{'url':'data:image/png;base64,'+b64}},{'type':'text','text':prompt}]}],
            timeout=150,
            extra_body={'chat_template_kwargs':{'enable_thinking':False}},
        )
    except Exception as e:
        raise HTTPException(502,'Gemma 8格辨識失敗: '+str(e))
    obj=extract_json(resp.choices[0].message.content)
    ban=obj.get('buyer_tax_id')
    if ban is not None:
        ban=re.sub(r'\D','',str(ban))
        if len(ban)!=8: ban=None
    digits=obj.get('digits') if isinstance(obj.get('digits'),list) else ([] if ban is None else list(ban))
    try: conf=float(obj.get('confidence') or 0)
    except Exception: conf=0.0
    return {'buyer_tax_id':ban,'digits':digits,'confidence':max(0,min(1,conf)),'model':'google/gemma-4-E4B-it','checksum_used':False}

@app.get('/health/v13')
async def health_v13():
    return {'status':'ok','version':'1.3.0','buyer_grid_endpoint':True,'checksum_mutation':False}
'''
open('/content/api-ocr-2025/custom_main.py','w',encoding='utf-8').write(custom)
compile(custom,'custom_main.py','exec')
print('✅ custom_main.py ready')


## 6. 啟動 api-ocr-2025 (:8080) 並做真正 VLM 健康檢查


In [ ]:
import os, subprocess, time, requests, json
log_path='/content/api-ocr-v13.log'
env=os.environ.copy()
env.update({
 'APIOCR_VLM_BACKEND':'local',
 'APIOCR_LOCAL_VLM_ENABLED':'true',
 'APIOCR_LOCAL_VLM_URL':'http://127.0.0.1:8001/v1',
 'APIOCR_LOCAL_VLM_MODEL':MODEL_ID,
 'APIOCR_LOCAL_VLM_TIMEOUT':'180',
 'APIOCR_VLM_ENABLED':'true',
 'APIOCR_USE_GPU':'false',
 'APIOCR_DESKEW':'true',
 'APIOCR_UPSCALE_MIN_SIDE':'1400',
 'APIOCR_MIN_CONFIDENCE':'0.20',
})
logf=open(log_path,'w')
api_proc=subprocess.Popen(
 ['python','-m','uvicorn','custom_main:app','--host','0.0.0.0','--port','8080','--workers','1'],
 cwd=REPO,env=env,stdout=logf,stderr=subprocess.STDOUT,text=True
)
for _ in range(120):
    try:
        r=requests.get('http://127.0.0.1:8080/health',timeout=2)
        if r.ok:
            print('✅ api-ocr-2025:',r.json());break
    except Exception: time.sleep(1)
else:
    logf.flush();print(open(log_path,encoding='utf-8',errors='ignore').read()[-8000:]);raise RuntimeError('api-ocr-2025 failed to start')

print('正在執行真正的文字＋影像 VLM 健康探針，第一次可能較慢…')
vr=requests.get('http://127.0.0.1:8080/health/vlm',timeout=200)
print('health/vlm HTTP',vr.status_code)
print(json.dumps(vr.json(),ensure_ascii=False,indent=2))
if vr.status_code!=200 or vr.json().get('status')!='ok':
    raise RuntimeError('Gemma 視覺健康檢查未通過；請先不要進入網站測試')


## 7. 建立 Cloudflare 臨時 HTTPS Tunnel

每次 Colab runtime 重啟 URL 會改變。把最後的 `COLAB_BACKEND_URL` 貼到 Tax AI V1.3。


In [ ]:
import os, subprocess, re, time, requests
cf='/content/cloudflared'
if not os.path.exists(cf):
    subprocess.run(['wget','-q','https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64','-O',cf],check=True)
    os.chmod(cf,0o755)
tunnel_log='/content/cloudflared-v13.log'
tf=open(tunnel_log,'w')
tunnel_proc=subprocess.Popen([cf,'tunnel','--url','http://127.0.0.1:8080','--no-autoupdate'],stdout=tf,stderr=subprocess.STDOUT,text=True)
public_url=None
for _ in range(60):
    time.sleep(1);tf.flush();txt=open(tunnel_log,encoding='utf-8',errors='ignore').read()
    m=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',txt)
    if m: public_url=m.group(0);break
if not public_url:
    print(open(tunnel_log,encoding='utf-8',errors='ignore').read()[-5000:]);raise RuntimeError('Tunnel URL not found')
print('\nCOLAB_BACKEND_URL =',public_url)
print('health =',requests.get(public_url+'/health',timeout=30).json())
print('v13 =',requests.get(public_url+'/health/v13',timeout=30).json())
print('\nTax AI V1.3: https://alexandroslee.github.io/test/tax-ai-v1/')


## 8. 直接在 Colab 用真實發票測試（建議先做）

上傳一張發票，這格會直接呼叫同一個 `/v1/invoice`，排除前端/Vercel 因素。


In [ ]:
from google.colab import files
import requests, json
uploaded=files.upload()
for name,data in uploaded.items():
    r=requests.post(
        public_url+'/v1/invoice',
        files={'file':(name,data,'image/jpeg')},
        data={'engine':'auto','slim':'false','include_image':'false'},
        timeout=295,
    )
    print('HTTP',r.status_code)
    try: print(json.dumps(r.json(),ensure_ascii=False,indent=2))
    except Exception: print(r.text[:5000])
